In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        x: Tensor of shape (batch_size, 784)
        returns: logits of shape (batch_size, 10)
        """
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        logits = self.fc3(x)  # logits (NO softmax here)
        return logits



In [ ]:
# “We train for 5 epochs using mini-batch gradient descent (batch size 64).”

# “Each iteration does forward pass → loss computation → backward pass → Adam parameter update.”

# “Validation uses torch.no_grad() and model.eval() to disable gradient tracking and dropout/batchnorm effects (if any).”


def train_and_validate(train_loader, val_loader, num_epochs=5, lr=0.001, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = DigitClassifier().to(device)

    # Loss + optimizer as specified
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        # -------------------
        # TRAIN
        # -------------------
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * x_batch.size(0)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == y_batch).sum().item()
            train_total += x_batch.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # -------------------
        # VALIDATION
        # -------------------
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(x_batch)
                loss = criterion(logits, y_batch)

                val_loss_sum += loss.item() * x_batch.size(0)
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total += x_batch.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    return model



In [ ]:
# Less vanishing gradient problem:
# Sigmoid/tanh saturate for large positive/negative inputs, making derivatives near zero. This causes gradients to vanish in deep networks. ReLU has derivative 1 for positive inputs, which helps gradients flow better.

# Computationally simpler and promotes sparsity:
# ReLU is just max(0, x) so it’s faster than sigmoid/tanh and often produces sparse activations (many zeros), which can make optimization easier and improve generalization.

In [ ]:


# “PyTorch’s autograd engine automatically computes gradients for tensors with requires_grad=True. During the forward pass, PyTorch builds a dynamic computation graph where each operation stores how it was computed (via grad_fn). When loss.backward() is called, autograd performs reverse-mode differentiation (backpropagation), applying the chain rule from the loss back to each parameter. The resulting gradients are stored in parameter.grad. The optimizer (Adam) then updates parameters using these gradients in optimizer.step(), and optimizer.zero_grad() clears gradients before the next iteration.”